In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("/code/src/")

In [ ]:
import json

In [ ]:
from data_processing.utils.geometry_utils import get_rotation_diff, get_3d_point_distance
from visualization.viser_visualization import get_traj_frames_data
from numpy.typing import NDArray
import numpy as np
from pathlib import Path

## Parameters

## Maincode

In [ ]:
src_dir = "/data/"
dataset_root = f"{src_dir}/datasets/processed"
scene_name = "backyard_sunny"
scene_data_file =  f"{dataset_root}/{scene_name}/scene_data.json"

In [ ]:
with open(scene_data_file, 'r') as f:
    scene_data = json.load(f)

In [ ]:
# def get_pose_weight_rot_distance(pose_a:NDArray, pose_b:NDArray, rot_w:float=0.1):
#     d_t = get_3d_point_distance(pose_a[:3, 3].tolist(), pose_b[:3, 3].tolist())
#     d_r = get_rotation_diff(pose_a[:3, :3], pose_b[:3,:3])

#     d = np.abs(d_t).item() + (rot_w * np.abs(d_r).item())

#     return d

In [ ]:
def compute_scene_diameter(camera_centers):
    pts = np.asarray(camera_centers)

    max_dist = 0.0
    for i in range(len(pts)):
        dists = np.linalg.norm(pts[i+1:] - pts[i], axis=1)
        if len(dists):
            max_dist = max(max_dist, dists.max())

    return float(max_dist)

In [ ]:
def compute_aabb_diagonal(camera_centers):
    pts = np.asarray(camera_centers)
    bbox_min = pts.min(axis=0)
    bbox_max = pts.max(axis=0)

    scene_scale = np.linalg.norm(bbox_max - bbox_min).item()

    return scene_scale

In [ ]:
def get_pose_t_distance(pose_a:NDArray,
                        pose_b:NDArray,
                        translation_scale:float=1.0, 
                        translation_weight:float=1.0):
    d_t = translation_weight * (get_3d_point_distance(pose_a[:3, 3].tolist(), pose_b[:3, 3].tolist()) / translation_scale)

    return np.abs(d_t).item()

In [ ]:
def get_pose_r_distance(pose_a:NDArray,
                        pose_b:NDArray,
                        rotation_scale:float=1.0,
                        rotation_weight:float=1.0):
    d_r = rotation_weight * (get_rotation_diff(pose_a[:3, :3], pose_b[:3,:3]) / rotation_scale)

    return np.abs(d_r).item()

In [ ]:
def get_pose_tr_distance(pose_a:NDArray, pose_b:NDArray, 
                                          translation_scale:float=1.0, rotation_scale:float=1,
                                          translation_weight:float=1.0, rotation_weight:float=1.0):
    
    d_t = get_pose_t_distance(pose_a=pose_a, 
                              pose_b=pose_b,
                              translation_scale=translation_scale, 
                              translation_weight=translation_weight)
    
    d_r = get_pose_r_distance(pose_a=pose_a, 
                              pose_b=pose_b,
                              rotation_scale=rotation_scale, 
                              rotation_weight=rotation_weight)

    d_tr = (0.5 * d_t) + (0.5 * d_r)

    return d_tr


In [ ]:
def get_pose_k_nearest_neighbor(pose, 
                                ref_poses,
                                pose_distance_fn, 
                                k=1):
    pose_distances = []
    for ref_pose in ref_poses:
        pose_distances.append(pose_distance_fn(pose, ref_pose))
    
    nearest_ids = np.argsort(pose_distances)[:k]
    distance = sum([pose_distances[nearest_id] for nearest_id in nearest_ids]) / k
    nearest_pose = ref_poses[nearest_ids[0]]

    return nearest_pose, distance, nearest_ids

def get_traj_directed_chamfer_distance(traj_a_poses, 
                                       traj_b_poses, 
                                       pose_distance_fn,
                                       k_neighbor_size=1):
    traj_a_distances = []
    traj_a_nearest_pose = []
    
    #TODO: This is brute force need to be changed to something smarted
    for pose_a in traj_a_poses:
        nearest_pose, distance, _ = get_pose_k_nearest_neighbor(pose_a, 
                                                                traj_b_poses, 
                                                                pose_distance_fn=pose_distance_fn,
                                                                k=k_neighbor_size)
        traj_a_distances.append(distance)
        traj_a_nearest_pose.append(nearest_pose)
    
    traj_a_to_b_distance = np.mean(traj_a_distances).item()

    return traj_a_to_b_distance


def get_traj_symmetric_chamfer_distance(traj_a_poses, 
                                        traj_b_poses, 
                                        pose_distance_fn,
                                        k_neighbor_size=1):
    
    traj_a_to_b_distance = get_traj_directed_chamfer_distance(traj_a_poses=traj_a_poses,
                                                                     traj_b_poses=traj_b_poses,
                                                                     pose_distance_fn=pose_distance_fn,
                                                                     k_neighbor_size=k_neighbor_size)
    
    traj_b_to_a_distance = get_traj_directed_chamfer_distance(traj_a_poses=traj_b_poses,
                                                                     traj_b_poses=traj_a_poses,
                                                                     pose_distance_fn=pose_distance_fn,
                                                                     k_neighbor_size=k_neighbor_size)

    return (0.5* traj_a_to_b_distance) + (0.5* traj_b_to_a_distance)


def get_trajectories_diff(scene_data, traj_a_name, traj_b_name, 
                          pose_distance_fn, 
                          k_neighbor_size=1,
                          pose_type="colmap_pose_c2w"):
    
    traj_a = get_traj_frames_data(scene_traj_data=scene_data["trajectories"], trajectory_name=traj_a_name,
                                cam_intrinsics_type="camera_intrinsic_colmap", c2w_pose_type=pose_type)

    traj_b = get_traj_frames_data(scene_traj_data=scene_data["trajectories"], trajectory_name=traj_b_name,
                                    cam_intrinsics_type="camera_intrinsic_colmap", c2w_pose_type=pose_type)
    
    traj_a_poses = [np.array(frame["pose_c2w"]) for frame in traj_a]
    traj_b_poses = [np.array(frame["pose_c2w"]) for frame in traj_b]

    traj_mean_dis = get_traj_directed_chamfer_distance(traj_a_poses=traj_a_poses, 
                                                        traj_b_poses=traj_b_poses,
                                                        pose_distance_fn=pose_distance_fn,
                                                        k_neighbor_size=k_neighbor_size)
            
    return traj_mean_dis

    

In [ ]:
trajectories =["orbit_inward_low", 
               "orbit_inward_mid",
               "orbit_inward_high",
               "traversal_forward_low",
               "traversal_backward_low",
               "traversal_left_low",
               "traversal_right_low",
               "orbit_inward_high",
               "panorama_360_station_a",
               "panorama_360_station_b",
               "panorama_360_station_c" ]

In [ ]:

trajectories = sorted(scene_data["trajectories"].keys())

In [ ]:
trajectories

In [ ]:
camera_centers = [np.array(frame_data["colmap_pose_c2w"])[:3, 3]  for traj in trajectories for frame_data in scene_data["trajectories"][traj]["frames"] if "colmap_pose_c2w" in frame_data]

In [ ]:
pts = np.array(camera_centers)

In [ ]:
pts.shape

In [ ]:
bbox_min = pts.min(axis=0)
bbox_max = pts.max(axis=0)

In [ ]:
scene_diameter = compute_scene_diameter(camera_centers)
bbox_diameter = compute_aabb_diagonal(camera_centers)

print(f"scene_diameter = {scene_diameter}")
print(f"bbox_diameter = {bbox_diameter}")

In [ ]:
trajectories_matrix = {}
k = 1
for traj_a in trajectories:
    trajectories_matrix[traj_a] = {}
    for traj_b in trajectories:
        trajectories_matrix[traj_a][traj_b] = {}
        traj_tr_distance = get_trajectories_diff(scene_data, 
                                              traj_a_name=traj_a, 
                                              traj_b_name=traj_b,
                                              k_neighbor_size=k,
                                              pose_distance_fn=lambda pose_a, pose_b:(
                                                 get_pose_tr_distance(pose_a=pose_a, 
                                                                      pose_b=pose_b, 
                                                                      translation_scale=bbox_diameter,
                                                                      rotation_scale=180)
                                                )       
                                            )
        trajectories_matrix[traj_a][traj_b]["directed_norm_chamfer_tr_distance"] = round(traj_tr_distance, 3)

        traj_t_distance = get_trajectories_diff(scene_data, 
                                              traj_a_name=traj_a, 
                                              traj_b_name=traj_b,
                                              k_neighbor_size=k,
                                              pose_distance_fn=lambda pose_a, pose_b:(
                                                 get_pose_t_distance(pose_a=pose_a, 
                                                                      pose_b=pose_b, 
                                                                      translation_scale=bbox_diameter)
                                                )       
                                            )
        trajectories_matrix[traj_a][traj_b]["directed_norm_chamfer_t_distance"] = round(traj_t_distance, 3)

        traj_r_distance = get_trajectories_diff(scene_data, 
                                              traj_a_name=traj_a, 
                                              traj_b_name=traj_b,
                                              k_neighbor_size=k,
                                              pose_distance_fn=lambda pose_a, pose_b:(
                                                 get_pose_r_distance(pose_a=pose_a, 
                                                                      pose_b=pose_b, 
                                                                      rotation_scale=180)
                                                )       
                                            )
        trajectories_matrix[traj_a][traj_b]["directed_norm_chamfer_r_distance"] = round(traj_r_distance, 3)


In [ ]:
print(json.dumps(trajectories_matrix, indent=4))